## Slowly Changing Dimension Type 2 (SCD2)

**SCD Type 2** mantiene el historial completo de cambios creando un nuevo registro cada vez que cambia un atributo. Cada registro tiene:
* `valido_desde` / `valido_hasta`: rango de vigencia
* `es_actual`: flag para identificar la versión actual

Cuando un dato cambia, se **expira** el registro anterior (ajustando `valido_hasta` y `es_actual = FALSE`) y se **inserta** uno nuevo con los datos actualizados.

---

## Implementación: Approach de 3 Pasos

La forma correcta de implementar SCD Type 2 es:

### **Paso 1: Staging** - Detectar cambios
Crear una vista temporal (o tabla persistente) que identifica qué registros son nuevos, modificados o sin cambios.
```sql
CREATE OR REPLACE TEMP VIEW staging_cuentas AS
SELECT c.*, 
       CASE 
         WHEN d.id_cuenta IS NULL THEN 'NUEVO'
         WHEN c.nombre_cuenta != d.nombre_cuenta 
              OR c.correo_electronico != d.correo_electronico THEN 'MODIFICADO'
         ELSE 'SIN_CAMBIO'
       END as accion
FROM cuentas c 
LEFT JOIN dim_cuentas d ON c.id_cuenta = d.id_cuenta AND d.es_actual = TRUE;
```

### **Paso 2: MERGE** - Expirar registros modificados
```sql
MERGE INTO dim_cuentas AS target
USING staging_cuentas AS s
ON target.id_cuenta = s.id_cuenta 
   AND target.es_actual = TRUE
   AND s.accion = 'MODIFICADO'
WHEN MATCHED THEN
    UPDATE SET
        target.valido_hasta = DATE(s.fecha_actualizacion) - INTERVAL 1 DAY,
        target.es_actual = FALSE;
```

**⚠️ Idempotencia**: Usamos `DATE(s.fecha_actualizacion)` en vez de `CURRENT_DATE()` para que la operación sea **idempotente**. Si usás `CURRENT_DATE()`, cada ejecución cambiaría `valido_hasta` incluso con los mismos datos de entrada.

### **Paso 3: INSERT** - Insertar nuevas versiones
```sql
INSERT INTO dim_cuentas (...)
SELECT ..., 
       DATE(s.fecha_actualizacion) as valido_desde,
       DATE '9999-12-31' as valido_hasta,
       TRUE as es_actual
FROM staging_cuentas s
WHERE s.accion IN ('NUEVO', 'MODIFICADO');
```

### **Ventajas**

* **Funciona correctamente**: Inserta nuevas versiones de registros modificados
* **Lógica clara**: Cada paso tiene una responsabilidad específica
* **Idempotente**: Ejecutar múltiples veces con los mismos datos produce el mismo resultado
* **Fácil de debuggear**: Podés inspeccionar el staging antes de aplicar cambios
* **Auditable**: Podés loggear qué registros cambiaron
* **Performance**: La lógica de detección se calcula una sola vez

---

## ⚠️ Error Común: ¿Por Qué NO Funciona el MERGE de 1 Paso?

Intuitivamente, podrías pensar en usar un solo `MERGE` así:
```sql
MERGE INTO dim_cuentas
USING source ON source.id_cuenta = dim_cuentas.id_cuenta
WHEN MATCHED AND accion = 'MODIFICADO' THEN UPDATE ... -- Expira el viejo
WHEN NOT MATCHED AND accion != 'SIN_CAMBIO' THEN INSERT ... -- Inserta el nuevo
```

**El problema**: Para registros **MODIFICADO**:
1. `WHEN MATCHED` se ejecuta ✅ → Expira el registro viejo
2. `WHEN NOT MATCHED` **NO se ejecuta** ❌ → La nueva versión nunca se inserta

**¿Por qué?** Porque `WHEN NOT MATCHED` solo se dispara cuando **no hay match en la condición del `ON`**. Para registros modificados, el `id_cuenta` **sí existe** en la tabla destino, entonces hay match y el `WHEN NOT MATCHED` nunca se ejecuta.

**Resultado**: Los registros modificados quedan solo expirados, sin versión actual. Por eso se necesita el approach de múltiples pasos.

In [0]:
-- ============================================
-- PASO 1: Crear tabla de staging con detección de cambios
-- ============================================
-- Usamos tabla persistente porque:
-- - Las vistas temporales no soportan parámetros en IDENTIFIER()
-- - Las tablas temporales no soportan CREATE ... AS en este entorno

CREATE OR REPLACE TABLE IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_cuentas') AS
SELECT 
    c.id_cuenta,
    c.nombre_cuenta,
    c.correo_electronico,
    c.fecha_creacion,
    c.fecha_actualizacion,
    CASE 
        WHEN d.id_cuenta IS NULL THEN 'NUEVO'
        WHEN c.nombre_cuenta != d.nombre_cuenta 
             OR c.correo_electronico != d.correo_electronico THEN 'MODIFICADO'
        ELSE 'SIN_CAMBIO'
    END as accion
FROM IDENTIFIER(:catalogo_origen || '.core_negocio_' || :alumno || '.cuentas') c
LEFT JOIN (
    SELECT * 
    FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_cuentas')
    WHERE es_actual = TRUE
) d ON c.id_cuenta = d.id_cuenta;

In [0]:
select accion, count(*) 
from IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_cuentas')
group by accion

In [0]:
-- ============================================
-- PASO 2: Expirar registros que han cambiado
-- ============================================
-- IMPORTANTE: Usamos DATE(s.fecha_actualizacion) para valido_hasta
-- en vez de CURRENT_DATE() para que la operación sea IDEMPOTENTE.
-- Si usás CURRENT_DATE(), cada ejecución cambiaría valido_hasta
-- incluso con los mismos datos de entrada.

MERGE INTO IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_cuentas') AS target
USING IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_cuentas') AS s
ON target.id_cuenta = s.id_cuenta 
   AND target.es_actual = TRUE
   AND s.accion = 'MODIFICADO'
WHEN MATCHED THEN
    UPDATE SET
        target.valido_hasta = DATE(s.fecha_actualizacion) - INTERVAL 1 DAY,
        target.es_actual = FALSE;

In [0]:
-- ============================================
-- PASO 3: Insertar nuevos registros (nuevos + modificados)
-- ============================================
-- NOTA: id_dim_cuenta se genera automáticamente (IDENTITY)
-- Usamos DATE(fecha_actualizacion) como valido_desde para idempotencia
INSERT INTO IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_cuentas')
    (id_cuenta, nombre_cuenta, correo_electronico, fecha_creacion, 
     fecha_actualizacion, valido_desde, valido_hasta, es_actual)
SELECT
    s.id_cuenta,
    s.nombre_cuenta,
    s.correo_electronico,
    s.fecha_creacion,
    s.fecha_actualizacion,
    DATE(s.fecha_actualizacion) as valido_desde,
    DATE '9999-12-31' as valido_hasta,
    TRUE as es_actual
FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.staging_cuentas') s
WHERE s.accion IN ('NUEVO', 'MODIFICADO');

In [0]:
-- Cuentas con más de una versión
SELECT *
FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_cuentas')
WHERE id_cuenta IN (
    SELECT id_cuenta
    FROM IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_cuentas')
    GROUP BY id_cuenta
    HAVING COUNT(*) > 1
)
order by id_cuenta